# 📐 VRAM 予算表

量子化/非量子化の内訳と TP=2 の余裕。


In [ ]:
import json
import os
import struct


def find_root():
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    # slug 一致を最優先。無ければ qwen を含むもの、最後に全候補。
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]


ROOT = find_root()



def header(path):
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        return json.loads(f.read(n).decode("utf-8", "replace"))


cfg = json.load(open(os.path.join(ROOT, "config.json")))
qc = cfg.get("quantization_config") or {}
ignore = set(qc.get("ignore") or [])

for name in ("model.safetensors", "model-mtp.safetensors"):
    p = os.path.join(ROOT, name)
    if not os.path.isfile(p):
        print("missing:", name)
        continue
    h = header(p)
    h.pop("__metadata__", None)
    q = 0
    nq = 0
    qn = 0
    for k, v in h.items():
        size = v["data_offsets"][1] - v["data_offsets"][0]
        base = k.rsplit(".", 1)[0]
        if base in ignore or any(base.startswith(ig) for ig in ignore):
            nq += size
            qn += 1
        elif k.endswith(("weight_packed", "weight_scale", "weight_shape",
                         "weight_zero_point", "weight_global_scale")):
            q += size
        else:
            nq += size
            qn += 1
    print(f"{name}: 量子化 {q / 2**30:.2f} GiB / 非量子化 {nq / 2**30:.2f} GiB "
          f"({qn} tensors)")

print("\n=== TP=2 の見積り ===")
W_TOTAL = 19.5
TP = 2
for util, cap in ((0.90, 16.0), (0.92, 16.0), (0.95, 16.0)):
    budget = cap * util
    weights = W_TOTAL / TP
    kv_tok = 2 * 16 * 4 * 256 * 2  # bytes/token
    print(f"  util={util}: budget/GPU={budget:.2f} GiB 重み={weights:.2f} GiB "
          f"残={budget - weights:.2f} GiB")

kv_tok = 2 * 16 * 4 * 256 * 2
for mlen in (8192, 16384, 32768, 65536):
    per_gpu = kv_tok * mlen / TP / 2**30
    print(f"  max_len={mlen:6d}: KV {per_gpu:.2f} GiB/GPU (offset 込み +20% で "
          f"{per_gpu * 1.2:.2f})")

print("\n=== 起動時ロード量 ===")
for name in ("model.safetensors", "model-mtp.safetensors"):
    p = os.path.join(ROOT, name)
    if os.path.isfile(p):
        print(f"  {name}: {os.path.getsize(p) / 2**30:.2f} GiB (disk)")

print("\n=== vision tower の重み（ignore 側）===")
h = header(os.path.join(ROOT, "model.safetensors"))
h.pop("__metadata__", None)
vis = sum(v["data_offsets"][1] - v["data_offsets"][0]
          for k, v in h.items() if k.startswith("model.visual."))
print(f"  visual 合計: {vis / 2**30:.2f} GiB "
      f"（--language-model-only で削減できる上限）")
